#### Imports

In [ ]:
from pflfunction.one_parameter_zeta_functions import L_functions
import sympy as sp
import latex2sympy2
import re
import json
import os
import flint
import ast

#### Creating your own input data

##### Utilities

In [ ]:
def build_PF(PF_arr,as_str=False):
    #
    # Given an array of matrices denoting generators of the PF ideal for some motive,
    # we turn these into a list of SymPy polynomials. Currently, this is designed
    # for one-parameter systems.
    # 
    PF_eqs = []
    for PF_eq in PF_arr:
        PF = 0
        for z_pow in range(len(PF_eq)):
            theta_poly = array_to_poly(PF_eq[z_pow],theta)
            PF += sp.factor(z**z_pow * theta_poly)
        if as_str:
            PF_eqs.append(str(PF))
        else:
            PF_eqs.append(PF)
    return PF_eqs

def array_to_poly(array,var):
    if array == []: # empty array
        return 1
    else: # if something
        poly = 0
        for i in range(len(array)):
            poly += array[i]*var**i
        return sp.factor(poly)

def univariate_poly_to_matrix(expr,var,make_int=False):
    if expr==None:
        return []
    poly = sp.Poly(expr,var)
    max_deg = max(poly.degree(var),0)
    mat = []
    if max_deg == 0:
        rat = sp.Poly.coeff_monomial(poly,1) # a/b has zero locus a-bz
        num = int(rat.numerator)
        negden = int(-rat.denominator)
        mat.append(num)
        mat.append(negden)
    else:
        if make_int:
            poly = sp.Poly(poly/sp.Poly(poly).content())
        for deg in range(max_deg+1):            
            mat.append(int(sp.Poly.coeff_monomial(poly,var**deg)))         
    return mat

def PF_arr_to_mat_arr(PF_arr):
    #
    # Given an array of PF equations [PF1, ..., PFn] written as SymPy polynomials,
    # we turn these into matrices for an easier storing format in input_data.txt.
    # In particular, given a PF equation, we return an array of arrays, each describing
    # powers in z, and powers in theta. Thus, the example of [[[0,0,0,0,1],[-120,-1250,-4375,-6250,-3125]]]
    # describes z^0(0*theta^0+0*theta^1+0*theta^2+0*theta^3+1*theta^4) 
    # + z^1(-120*theta^0-1250*theta -4375*theta^2 -6250*theta^3-3125*theta^4), i.e. the mirror quintic PF.
    # We currently assume these are one-parameter systems.
    #
    mat_arr = []
    for PF in PF_arr:
        PF_to_add = []
        max_deg = sp.Poly(PF).degree(z)
        for deg in range(max_deg+1):
            # need make_int=False as this goes degree by degree; to remove any overall scaling, 
            # would want to act on each PF individually.
            PF_to_add.append(univariate_poly_to_matrix(PF.coeff(z,deg),theta,make_int=False)) 
        mat_arr.append(PF_to_add)
    return mat_arr

def PF_arr_to_str(PF_arr):
    mat_arr = PF_arr_to_mat_arr(PF_arr)
    s = json.dumps(mat_arr, separators=(',', ':'))
    return s

##### Picard-Fuchs Equation

We discuss how to denote the Picard-Fuchs operator in a readable format for both the function L_functions, as well as how to store it in input_data.txt. In particular, we show this via means of the Picard-Fuchs equation for the mirror quintic.

Data in input_data.txt cannot have spaces.

In [ ]:
theta = sp.symbols('theta') # defining variables
z = sp.symbols('z')

In [ ]:
tex = r'\theta^4-5 z(5\theta+1)(5\theta+2)(5\theta+3)(5\theta+4)'
tex = re.sub(r'(?<=[\d)])([a-zA-Z])', r'*\1', tex) # turns e.g. 11/16z into 11/16*z

In [ ]:
operator = sp.factor(latex2sympy2.latex2sympy(tex),z) # turns this into a SymPy polynomial
operator # this operator is what can be input into the first argument of L_functions

In [ ]:
PF_arr_to_mat_arr([operator]) # this is how we store the operator in inputs.txt. The format is described in 
# utilities above.

In [ ]:
build_PF(PF_arr_to_mat_arr([operator]),as_str=True)[0]

In [ ]:
PF_arr_to_str([operator]) # when storing in input_data.txt, we cannot have empty spaces.

##### Singularities

For the mirror quintic, there is a conifold singularity at $z=1/5^5$, there are no apparent singularities, and there are no other singularities (K points, MUM points, etc.). We cannot deal with singularities at infinity with respect to our expansion. Singularities are read into L_functions() as SymPy polynomials whose zero locus describes the singularity; we always make these polynomials integral by clearing denominators. If there is no singularity of a given type, we denote this by 1.

In inpud_data.txt, we denote singularities together as [coni, app, rest]. If a given type of singularity isn't present, we simply put in '[]'. Else, we make a list [sing1, sing2, ...], where each singularity is described by a univariate matrix in the same style as above; namely, [[[1,-3125]],[],[]] means there are no apparent singularities, no other singularities, and there is one conifold singularity at the zero locus of 1*z^0-3125*z^1.

Data in input_data.txt cannot have spaces.

In [ ]:
coni=sp.Poly(1-5**5*z)
app=None
rest=None

In [ ]:
json.dumps([[univariate_poly_to_matrix(coni,z,make_int=True)],univariate_poly_to_matrix(app,z,True),univariate_poly_to_matrix(rest,z,True)], separators=(',', ':'))

In [ ]:
print(array_to_poly(univariate_poly_to_matrix(coni,z,make_int=True),z))
print(array_to_poly(univariate_poly_to_matrix(app,z,make_int=True),z))
print(array_to_poly(univariate_poly_to_matrix(rest,z,make_int=True),z))

##### Supplementary Data

For the mirror quintic (and threefolds in general), we need the supplementary data of $\alpha_1,\alpha_2,\alpha_3$ for the $U_p(0)$ matrix. Fortunately, two of these vanish, and so we are left with one constant to feed in, which is the Euler characteristic / the intersection number. This is fed in as [Euler characteristic, intersection number]

For the mirror quintic, this is fed in as [-200,5] in the input data.

For K3 surfaces, all of the values $\alpha_i$ are zero, and so one may include e.g. [0,1] in the input data.

The data that is fed into the actual code directly actually is the Euler characteristic of the mirror manifold / intersection number of the original manifold, i.e. - Euler characteristic of the original / intersection number of the original.

In [ ]:
chiratio = 40

#### Running the code directly

In [ ]:
theta = sp.symbols('theta')
z = sp.symbols('z')

In [ ]:
print(L_functions(theta**4-5*z*(5*theta+1)*(5*theta+2)*(5*theta+3)*(5*theta+4),1,7,1-3125*z,1,1,40,"mirror_quintic"))

In [ ]:
print(L_functions(theta**3-2*z*(1+2*theta)*(2+5*theta*(1+theta))+64*z**2*(1+theta)**3,1.5,7,(-1+4*z)*(-1+16*z),1,1,0,"HVK3"))

In [ ]:
print(L_functions(theta**4-225*z**3*(theta+1)**2*(theta+2)**2+z**2*(theta+1)**2*(259*theta**2+518*theta+285)-z*(35*theta**4+70*theta**3+63*theta**2+28*theta+5),3,19,(1-z)*(1-9*z)*(1-25*z),1,1,flint.fmpq(2,3),"4.3.1"))

In [ ]:
print(L_functions(theta**4-944784*z**3*(theta+1)*(theta+2)*(2*theta+1)*(2*theta+5)-2916*z**2*(theta+1)**2*(20*theta**2+40*theta+17)-18*z*(6*theta**4+12*theta**3+3*theta**2-3*theta-1),5,7,1-324*z,1,1+108*z,flint.fmpq(-4,3),"4.3.7"))

In [ ]:
print(L_functions(theta**4-944784*z**3*(theta+1)*(theta+2)*(2*theta+1)*(2*theta+5)-2916*z**2*(theta+1)**2*(20*theta**2+40*theta+17)-18*z*(6*theta**4+12*theta**3+3*theta**2-3*theta-1),5,7,1-324*z,1,1+108*z,flint.fmpq(-4,3),"4.3.7",pacc_init=50,nadd=300))